# ⚡ Notebook 1: Apache Spark Architecture & Deep Internals
### *From Cluster Mechanics to DAGs, Catalyst Optimization, and Tungsten Execution*

> **Companion Video Reference:** [YouTube: PySpark Tutorial | Full Course (From Zero to Pro!)](https://www.youtube.com/watch?v=94w6hPk7nkM) by Ansh Lamba & [Advancing Analytics Spark Internals](https://www.youtube.com/@AdvancingAnalytics)

---

## 🎯 What Will You Learn in This Notebook?

Apache Spark is the undisputed industry standard for big data processing, distributed data engineering, and machine learning pipelines. 

While many tutorials only teach surface-level syntax (`df.select()`, `df.filter()`), **real-world Spark engineering requires understanding how Spark works under the hood**. Without internal knowledge, queries that take 10 seconds can easily take 4 hours or crash with `OutOfMemoryError`.

In this notebook, we explore:
1. **The Evolution of Big Data:** Why MapReduce lost and why Spark won.
2. **The Distributed Cluster Architecture:** Driver Node, Worker/Executor Nodes, and Cluster Managers.
3. **The Execution Hierarchy:** Applications $\rightarrow$ Jobs $\rightarrow$ Stages $\rightarrow$ Tasks.
4. **Resilient Distributed Datasets (RDDs) & Lineage Graphs:** Fault tolerance without disk replication.
5. **DAG (Directed Acyclic Graph) & Lazy Evaluation:** Why Spark waits before computing.
6. **Narrow vs. Wide Transformations:** The network **Shuffle barrier** and stage splitting.
7. **The Catalyst Optimizer:** Logical Plans, Optimization Rules (Predicate Pushdown, Column Pruning), and Physical Plans.
8. **Project Tungsten:** Off-heap memory layout (`UnsafeRow`) and Whole-Stage Code Generation.
9. **Hands-On Inspection:** Initializing `SparkSession`, triggering actions, and deciphering `.explain(extended=True)`.

---


## 🏛️ 1. The Evolution: Why MapReduce Failed & Why Spark Won

In the early 2000s, Google published the MapReduce paper, and Apache Hadoop was born. However, Hadoop MapReduce had a fundamental bottleneck:

```
HADOOP MAPREDUCE (Disk-Bound):
[Input Data] ──▶ (Map) ──▶ [Write to Disk/HDFS] ──▶ (Shuffle/Network) ──▶ (Reduce) ──▶ [Write to Disk/HDFS]
                                ▲                                                          │
                                └── Iterative ML / Multi-step SQL repeats disk write cycle ─┘
```

Every single step in MapReduce forced data to be serialized to hard disks (HDFS). For iterative algorithms (like Gradient Descent in Machine Learning or multi-step SQL queries), **90% of execution time was wasted doing slow disk I/O and network serialization**.

### The Spark Revolution (2014)
UC Berkeley's AMPLab created Apache Spark with two foundational breakthroughs:
1. **In-Memory Computing:** Data is kept in RAM across multiple steps. Disk is only used if RAM overflows (spill).
2. **Directed Acyclic Graph (DAG) & Lineage:** Instead of writing intermediate results to disk for fault tolerance, Spark remembers the **recipe (lineage)** of how each piece of data was computed. If an executor machine crashes, Spark simply re-runs that recipe for the lost partition!

Result: Spark is **up to 100x faster than Hadoop MapReduce** for iterative workloads.


## 🧩 2. Spark Cluster Architecture

Spark follows a classic **Master-Worker** distributed computing architecture:

```
                            ┌──────────────────────────────────────────────┐
                            │                 DRIVER NODE                  │
                            │  ┌────────────────────────────────────────┐  │
                            │  │              SparkSession              │  │
                            │  ├────────────────────────────────────────┤  │
                            │  │  DAGScheduler    │    TaskScheduler    │  │
                            │  └──────────────────┬─────────────────────┘  │
                            └─────────────────────┼────────────────────────┘
                                                  │ (Negotiates Resources)
                                                  ▼
                            ┌──────────────────────────────────────────────┐
                            │               CLUSTER MANAGER                │
                            │     (Standalone / YARN / Kubernetes)         │
                            └───────┬──────────────────────────────┬───────┘
                                    │ (Spawns & Monitors)          │
                   ┌────────────────┴───────────────┐              │
                   ▼                                                ▼
     ┌───────────────────────────┐                    ┌───────────────────────────┐
     │      WORKER NODE 1        │                    │      WORKER NODE 2        │
     │  ┌─────────────────────┐  │                    │  ┌─────────────────────┐  │
     │  │      EXECUTOR 1     │  │                    │  │      EXECUTOR 2     │  │
     │  │  [Core 1]  [Core 2] │  │                    │  │  [Core 1]  [Core 2] │  │
     │  │  [  JVM Heap RAM  ] │  │                    │  │  [  JVM Heap RAM  ] │  │
     │  │  [ Off-Heap Memory] │  │                    │  │  [ Off-Heap Memory] │  │
     │  │  Task 1     Task 2  │  │                    │  │  Task 3     Task 4  │  │
     │  └─────────────────────┘  │                    │  └─────────────────────┘  │
     └───────────────────────────┘                    └───────────────────────────┘
```

### The Core Roles:
1. **Driver Node (The Brain):**
   - The machine where your Python script starts and where `SparkSession` lives.
   - Converts user code into a **DAG (Directed Acyclic Graph)**.
   - `DAGScheduler`: Breaks the DAG into **Stages** based on shuffle boundaries.
   - `TaskScheduler`: Breaks stages into **Tasks** and dispatches them to Executors.
   - Coordinates cluster progress and collects final results.

2. **Cluster Manager (The Resource Allocator):**
   - Allocates hardware (CPU cores, RAM) across the cluster.
   - Supported managers: **Kubernetes** (modern standard), **YARN** (Hadoop standard), **Spark Standalone** (built-in), and **Local Mode** (runs everything on your laptop via threads).

3. **Executors (The Muscle):**
   - Dedicated JVM worker processes running on Worker Nodes.
   - Each executor is allocated a set number of CPU cores (slots) and RAM.
   - **Cores / Slots:** Determine how many tasks an executor can run concurrently (e.g. 4 cores = 4 concurrent tasks).
   - Responsible for running assigned tasks and storing cached data partitions in RAM or disk.


## 🪜 3. The Execution Hierarchy: Application ➔ Job ➔ Stage ➔ Task

Understanding this 4-tier hierarchy is essential for reading the Spark UI and debugging production bottlenecks:

```
┌────────────────────────────────────────────────────────────────────────┐
│ APPLICATION: One complete Spark program (from SparkSession creation    │
│              to spark.stop()).                                         │
└───────────────────────────────────┬────────────────────────────────────┘
                                    │ (Triggers when an ACTION is called)
                                    ▼
┌────────────────────────────────────────────────────────────────────────┐
│ JOB: Triggered every time you call an ACTION (e.g., .count(), .show(), │
│      .write.parquet(), .collect()).                                    │
└───────────────────────────────────┬────────────────────────────────────┘
                                    │ (Split by SHUFFLE boundaries)
                                    ▼
┌────────────────────────────────────────────────────────────────────────┐
│ STAGE: A pipeline of Narrow transformations that can run together      │
│        without transferring data across the network.                   │
└───────────────────────────────────┬────────────────────────────────────┘
                                    │ (1 Task per Data Partition)
                                    ▼
┌────────────────────────────────────────────────────────────────────────┐
│ TASK: The smallest unit of execution. Runs on a single CPU core        │
│       against a single partition of data in an Executor's JVM.         │
└────────────────────────────────────────────────────────────────────────┘
```

> 💡 **Golden Rule:** If a DataFrame has 200 partitions, a Stage will create exactly **200 Tasks**. If your cluster has 50 CPU cores, those 200 tasks will run in 4 consecutive waves of 50!


## 🔄 4. Transformations vs. Actions & Lazy Evaluation

Spark divides all DataFrame operations into two strict categories:

### 1. Transformations (Lazy)
- Operations that produce a **new DataFrame** from an existing one (e.g., `select()`, `filter()`, `withColumn()`, `groupBy()`, `join()`).
- **They do NOT compute anything immediately!**
- When you call `df.filter(...)`, Spark simply appends that operation to its internal plan (DAG). You can chain 50 transformations together and Spark won't touch a single byte of data.

### 2. Actions (Eager)
- Operations that require data to be computed and returned to the driver, displayed, or written to storage:
  - `show()`, `head()`, `first()`
  - `count()`
  - `collect()` *(Caution: brings all distributed data to driver RAM!)*
  - `write.csv()`, `write.parquet()`
- **Calling an Action triggers the execution of the entire upstream DAG!**

### 💡 Why Lazy Evaluation?
Why doesn't Spark execute line-by-line like Pandas or standard Python?
1. **Whole-Query Optimization:** If you filter data on line 10 (`df.filter(col("country") == "US")`), Spark's Catalyst Optimizer pushes that filter down to the storage layer on line 1 (**Predicate Pushdown**). Only US rows are ever loaded into RAM!
2. **Column Pruning:** If your CSV has 100 columns but you only select 2, Spark only reads those 2 columns from disk.
3. **Pipelining:** Multiple transformations (e.g. `map` followed by `filter`) are compiled into a single bytecode loop, avoiding intermediate memory allocations.


## 🔀 5. Narrow vs. Wide Transformations & The Shuffle Barrier

This is the **single most important concept** in distributed computing:

```
NARROW TRANSFORMATION (Fast, No Network Shuffle):
Partition 1 [Data] ──▶ (filter / map) ──▶ Partition 1 [Data]
Partition 2 [Data] ──▶ (filter / map) ──▶ Partition 2 [Data]
Partition 3 [Data] ──▶ (filter / map) ──▶ Partition 3 [Data]
* Each output partition depends on exactly ONE input partition.
* Runs in-memory inside the same executor. No network traffic!

=====================================================================

WIDE TRANSFORMATION (Slow, Requires SHUFFLE):
Partition 1 [Data] ──┐ ┌──▶ Partition 1 (Group A)
Partition 2 [Data] ──┼─┼──▶ Partition 2 (Group B)
Partition 3 [Data] ──┘ └──▶ Partition 3 (Group C)
* Each output partition depends on data from MULTIPLE input partitions.
* Requires the SHUFFLE: writing data to disk, transmitting over network, and re-sorting!
* A Wide transformation forces Spark to create a NEW STAGE boundary!
```

| Type | Examples | Network Shuffle? | Stage Boundary? |
| :--- | :--- | :--- | :--- |
| **Narrow** | `select()`, `filter()`, `withColumn()`, `drop()`, `map()`, `union()` | ❌ No | ❌ Kept in same stage |
| **Wide** | `groupBy()`, `join()`, `distinct()`, `repartition()`, `cube()`, `rollup()` | ✅ **Yes (Heavy)** | ✅ **Splits into New Stage** |


## 🧠 6. The Catalyst Optimizer & Project Tungsten

How does Spark turn high-level Python code into ultra-fast machine bytecode?

### 6.1 The Catalyst Optimizer Pipeline

```
 [ Python Code / SQL Query ]
              │
              ▼
  1. Unresolved Logical Plan     (Validates syntax, column names still unverified)
              │
              ▼ (Catalyst Catalog: Checks table & column schemas)
  2. Analyzed Logical Plan       (Verified columns and data types)
              │
              ▼ (Logical Optimizations: Predicate pushdown, projection pruning, constant folding)
  3. Optimized Logical Plan      (Most efficient relational algebraic plan)
              │
              ▼ (Physical Planning: Cost-Based Model chooses algorithms, e.g. BroadcastHashJoin vs SortMergeJoin)
  4. Physical Plans
              │
              ▼ (Whole-Stage Code Generation: Compiles Java Bytecode directly to CPU registers)
  5. Executable RDD Code
```

### 6.2 Project Tungsten: Sub-Millisecond Hardware Acceleration
Python and JVM objects have heavy memory overhead (a 4-byte integer in Java can take 16-24 bytes of memory due to object headers).
- **Off-Heap Memory Management:** Tungsten allocates raw C-style memory buffers using `sun.misc.Unsafe`, bypassing Java Garbage Collection (GC) pauses completely!
- **`UnsafeRow` Binary Format:** Compact binary format stored in CPU L1/L2 caches.
- **Whole-Stage CodeGen:** Instead of using virtual function dispatches for each row, Tungsten generates clean Java bytecode that compiles down to raw CPU assembly loops!


## 💻 7. Hands-On Lab: Initializing Spark & Inspecting Internals

Now let's launch a local `SparkSession` and physically inspect these internal concepts:
- Initializing with fine-tuned configs
- Creating a distributed DataFrame
- Examining the DAG and Catalyst execution plans using `.explain(extended=True)`
- Inspecting partition allocations


In [1]:
import os
import sys
from pathlib import Path

# Configure environment: reduce verbosity and set log level
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

from pyspark.sql import SparkSession
from pyspark.sql.functions import col, when, lit, avg, sum

# Initialize SparkSession in local mode (using all available CPU threads)
print("⏳ Starting SparkSession...")
spark = SparkSession.builder \
    .appName("Spark_Architecture_and_Internals") \
    .master("local[*]") \
    .config("spark.sql.shuffle.partitions", "4") \
    .config("spark.driver.memory", "2g") \
    .config("spark.ui.enabled", "false") \
    .getOrCreate()

# Set log level to WARN to keep output clean and readable
spark.sparkContext.setLogLevel("WARN")

print("✅ SparkSession successfully initialized!")
print(f"  Spark Version:     {spark.version}")
print(f"  Master:            {spark.sparkContext.master}")
print(f"  Default Parallelism (CPU Cores): {spark.sparkContext.defaultParallelism}")


⏳ Starting SparkSession...


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/09/22 15:17:22 WARN Utils: Your hostname, Prafull-Mac.local, resolves to a loopback address: 127.0.0.1; using 192.168.29.23 instead (on interface en0)
26/09/22 15:17:22 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
/Users/prafullsaxena/Desktop/Learning/Machine Learning/milvus/.venv/lib/python3.12/site-packages/pyspark/testing/utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
26/09/22 15:17:24 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


✅ SparkSession successfully initialized!
  Spark Version:     4.2.0
  Master:            local[*]
  Default Parallelism (CPU Cores): 8


### Creating a Sample Dataset & Inspecting Partitions
Let's create a distributed dataset of developer transactions across different regions and examine how Spark partitions it across memory:


In [2]:
# Sample data: Developer platform transactions
data = [
    (101, "Alice", "Python", 450.0, "US"),
    (102, "Bob", "Java", 300.0, "US"),
    (103, "Charlie", "Scala", 600.0, "EU"),
    (104, "David", "Python", 200.0, "APAC"),
    (105, "Emma", "Go", 520.0, "EU"),
    (106, "Frank", "Python", 410.0, "US"),
    (107, "Grace", "Java", 290.0, "APAC"),
    (108, "Henry", "Rust", 750.0, "EU"),
    (109, "Ivy", "Go", 480.0, "US"),
    (110, "Jack", "Rust", 820.0, "APAC"),
]

columns = ["id", "name", "skill", "billable_rate", "region"]

# Create distributed DataFrame
df = spark.createDataFrame(data, columns)

print(f"✅ DataFrame created!")
print(f"  Total Partitions: {df.rdd.getNumPartitions()}")
df.show()


✅ DataFrame created!
  Total Partitions: 8


+---+-------+------+-------------+------+
| id|   name| skill|billable_rate|region|
+---+-------+------+-------------+------+
|101|  Alice|Python|        450.0|    US|
|102|    Bob|  Java|        300.0|    US|
|103|Charlie| Scala|        600.0|    EU|
|104|  David|Python|        200.0|  APAC|
|105|   Emma|    Go|        520.0|    EU|
|106|  Frank|Python|        410.0|    US|
|107|  Grace|  Java|        290.0|  APAC|
|108|  Henry|  Rust|        750.0|    EU|
|109|    Ivy|    Go|        480.0|    US|
|110|   Jack|  Rust|        820.0|  APAC|
+---+-------+------+-------------+------+



### Observing Lazy Evaluation in Action
Notice that executing the cell below is instantaneous because Spark is only registering the transformations in its logical plan. No rows are scanned or aggregated yet!


In [3]:
# Step 1: Narrow Transformation (filter & withColumn)
filtered_df = df.filter(col("billable_rate") >= 400.0) \
                .withColumn("tier", when(col("billable_rate") >= 600.0, lit("Premium")).otherwise(lit("Standard")))

# Step 2: Wide Transformation (groupBy and agg - triggers network shuffle!)
summary_df = filtered_df.groupBy("region", "tier") \
                        .agg(
                            sum("billable_rate").alias("total_rate"),
                            avg("billable_rate").alias("avg_rate")
                        )

print("⚡ Transformations registered in Spark's Catalyst engine!")
print("Notice: Not a single byte has been computed yet (Lazy Evaluation)!")


⚡ Transformations registered in Spark's Catalyst engine!
Notice: Not a single byte has been computed yet (Lazy Evaluation)!


### Triggering an Action & Inspecting the Catalyst Plans
Now we call `.show()`, which is an **Action**.
Before Spark computes the result, let's call `.explain(extended=True)` to see all 4 stages of the Catalyst Optimizer:
1. **Parsed Logical Plan**
2. **Analyzed Logical Plan**
3. **Optimized Logical Plan** (Notice how the filter is pushed down and columns are pruned!)
4. **Physical Plan** (Shows WholeStageCodegen, HashAggregate, and Exchange/Shuffle)


In [ ]:
# Inspect the complete Catalyst Optimization Pipeline
print("🔍 --- CATALYST OPTIMIZATION PIPELINE ---")
summary_df.explain(extended=True)

print("\n🚀 --- TRIGGERING ACTION: COMPUTING FINAL RESULTS ---")
summary_df.show()


🔍 --- CATALYST OPTIMIZATION PIPELINE ---
== Parsed Logical Plan ==
'Aggregate ['region, 'tier], ['region, 'tier, 'sum('billable_rate) AS total_rate#22, 'avg('billable_rate) AS avg_rate#23]
+- Project [id#0L, name#1, skill#2, billable_rate#3, region#4, CASE WHEN (billable_rate#3 >= 600.0) THEN Premium ELSE Standard END AS tier#21]
   +- Filter (billable_rate#3 >= 400.0)
      +- LogicalRDD [id#0L, name#1, skill#2, billable_rate#3, region#4], false

== Analyzed Logical Plan ==
region: string, tier: string, total_rate: double, avg_rate: double
Aggregate [region#4, tier#21], [region#4, tier#21, sum(billable_rate#3) AS total_rate#22, avg(billable_rate#3) AS avg_rate#23]
+- Project [id#0L, name#1, skill#2, billable_rate#3, region#4, CASE WHEN (billable_rate#3 >= 600.0) THEN Premium ELSE Standard END AS tier#21]
   +- Filter (billable_rate#3 >= 400.0)
      +- LogicalRDD [id#0L, name#1, skill#2, billable_rate#3, region#4], false

== Optimized Logical Plan ==
Aggregate [region#4, tier#21], [re

26/09/22 15:44:21 WARN HeartbeatReceiver: Removing executor driver with no recent heartbeats: 212508 ms exceeds timeout 120000 ms
26/09/22 15:44:21 WARN SparkContext: Killing executors is not supported by current scheduler.
26/09/22 15:44:26 ERROR Inbox: Ignoring error
org.apache.spark.SparkException: Exception thrown in awaitResult: 
	at org.apache.spark.util.SparkThreadUtils$.awaitResult(SparkThreadUtils.scala:70)
	at org.apache.spark.util.SparkThreadUtils$.awaitResult(SparkThreadUtils.scala:44)
	at org.apache.spark.util.ThreadUtils$.awaitResult(ThreadUtils.scala:359)
	at org.apache.spark.rpc.RpcTimeout.awaitResult(RpcTimeout.scala:75)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRefByURI(RpcEnv.scala:102)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRef(RpcEnv.scala:110)
	at org.apache.spark.util.RpcUtils$.makeDriverRef(RpcUtils.scala:34)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.driverEndpoint$lzycompute(BlockManagerMasterEndpoint.scala:132)
	at org.apache.spark.stora

### Clean Session Shutdown
Always stop the `SparkSession` when your workload finishes to release the JVM heap and background thread pools:


In [5]:
# Cleanly shut down the SparkContext
spark.stop()
print("✅ SparkSession cleanly terminated. Resources released!")


✅ SparkSession cleanly terminated. Resources released!


## 📖 Key Takeaways: Spark Architecture Cheat Sheet

| Component | Responsibility | Why It Matters to You |
| :--- | :--- | :--- |
| **Driver** | Maintains SparkSession, creates DAG, schedules stages/tasks. | If Driver runs out of RAM (`Driver OOM`), avoid calling `.collect()` on large datasets. |
| **Executors** | JVM worker processes that run tasks and hold cached data in RAM. | Increasing executor count scales your compute horizontally. |
| **Partition** | Atomic chunk of data (usually 128 MB on disk). | 1 Partition = 1 Task = 1 CPU Core at a time. Partitions govern concurrency! |
| **Narrow Transf.** | 1-to-1 data flow (e.g. `select`, `filter`). | Ultra-fast. No network transfer, pipelined in CPU cache. |
| **Wide Transf.** | N-to-N data flow (e.g. `groupBy`, `join`). | Triggers **Shuffle**. Writes to disk, transmits over network. Primary source of bottlenecks. |
| **Catalyst** | Query planner & optimizer. | Rewrites your queries to be optimal (e.g. Predicate Pushdown). |
| **Tungsten** | Memory management & bytecode generation. | Stores data off-heap in binary `UnsafeRow` to bypass Java garbage collection pauses. |

---
**Next Step:** Move to **`02_dataframes_and_structured_api.ipynb`** to master the PySpark DataFrame API, schemas, and file ingestion!
